<a href="https://colab.research.google.com/github/9ecolla/Data-Science-/blob/main/FCS/rag_demo_dnd_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Короткое демо RAG в Google Colab

В этом ноутбуке показан минимальный RAG-пайплайн на примере PDF-файла с книгой правил DnD 5e.

Пайплайн состоит из этапов:

1. загрузка PDF;
2. извлечение текста;
3. разбиение текста на чанки;
4. построение эмбеддингов;
5. создание FAISS-индекса;
6. поиск релевантных фрагментов;
7. генерация ответа с использованием найденного контекста.

Важно: используйте только файл, который у вас есть на законных основаниях. Не публикуйте извлечённый текст или индекс с полным содержанием книги.

## 1. Установка библиотек

Устанавливаем библиотеки для чтения PDF, векторного поиска, работы с OpenAI-совместимым API и fallback-вариантов через Hugging Face.

In [ ]:
# pypdf извлекает текст из PDF.
# faiss-cpu выполняет быстрый поиск ближайших векторов.
# openai используется для OpenAI-совместимого API, например OpenRouter.
# sentence-transformers нужен для локальных эмбеддингов через Hugging Face.
# transformers и accelerate нужны для локальной генерации через Hugging Face.

!pip -q install pypdf faiss-cpu openai sentence-transformers transformers accelerate

## 2. Импорты и конфигурация

Подключаем библиотеки и задаём основные параметры пайплайна: путь к PDF, размер чанков, модели эмбеддингов и генерации.

In [ ]:
import os
import re
import json
import numpy as np
import faiss

from typing import List, Dict, Any
from getpass import getpass
from pypdf import PdfReader
from openai import OpenAI

In [ ]:
# Путь, по которому будет сохранён загруженный PDF.
PDF_PATH = "/content/dnd_players_handbook.pdf"

# Размер одного текстового фрагмента в символах.
# Для короткого демо используется простой character-based chunking.
CHUNK_SIZE = 1200

# Перекрытие между соседними чанками помогает сохранить смысл на границах фрагментов.
CHUNK_OVERLAP = 200

# Количество фрагментов, которые retrieval вернёт для одного вопроса.
TOP_K = 5

# Модель генерации через OpenRouter. Её можно заменить на другую доступную модель.
GENERATION_MODEL_OPENROUTER = "openai/gpt-4o-mini"

# Модель эмбеддингов через OpenRouter.
EMBEDDING_MODEL_OPENROUTER = "openai/text-embedding-3-small"

# Fallback-модель эмбеддингов через Hugging Face.
EMBEDDING_MODEL_HF = "sentence-transformers/all-MiniLM-L6-v2"

## 3. Загрузка PDF в Colab

После запуска ячейки появится форма загрузки файла. Загруженный PDF будет переименован в стандартный путь `PDF_PATH`, чтобы остальные ячейки работали без изменений.

In [ ]:
from google.colab import files

# Загружаем PDF-файл в окружение Colab.
uploaded = files.upload()

# Берём первый загруженный файл и сохраняем его под ожидаемым именем.
for filename in uploaded.keys():
    os.rename(filename, PDF_PATH)
    print(f"Файл сохранён как: {PDF_PATH}")
    break

## 4. Извлечение текста из PDF

PDF читается постранично. Для каждой страницы сохраняются номер страницы и очищенный текст. Номер страницы понадобится для отображения источников в ответе.

In [ ]:
def extract_text_from_pdf(pdf_path: str) -> List[Dict[str, Any]]:
    """
    Извлекает текст из PDF постранично.

    Возвращает список словарей вида:
    {
        "page": номер_страницы,
        "text": текст_страницы
    }
    """
    reader = PdfReader(pdf_path)
    pages = []

    for page_index, page in enumerate(reader.pages):
        # extract_text может вернуть None, если текст на странице не распознан.
        text = page.extract_text() or ""

        # Упрощаем пробелы и переносы строк, чтобы дальнейшая обработка была стабильнее.
        text = re.sub(r"\s+", " ", text).strip()

        # Пустые страницы не добавляем в корпус.
        if text:
            pages.append({
                "page": page_index + 1,
                "text": text
            })

    return pages


pages = extract_text_from_pdf(PDF_PATH)

print(f"Извлечено страниц с текстом: {len(pages)}")
print("Пример текста с первой найденной страницы:")
print(pages[0]["text"][:1000])

## 5. Разбиение текста на чанки

LLM не получает всю книгу целиком. Текст заранее делится на небольшие перекрывающиеся фрагменты, а затем для каждого вопроса выбираются только самые релевантные из них.

In [ ]:
def chunk_text(
    pages: List[Dict[str, Any]],
    chunk_size: int = 1200,
    overlap: int = 200
) -> List[Dict[str, Any]]:
    """
    Разбивает текст страниц на перекрывающиеся чанки.

    Каждый чанк хранит сам текст и номер страницы, с которой он был получен.
    """
    chunks = []

    for page in pages:
        text = page["text"]
        start = 0

        while start < len(text):
            end = start + chunk_size
            chunk = text[start:end].strip()

            # Слишком короткие фрагменты обычно дают мало полезного контекста.
            if len(chunk) > 100:
                chunks.append({
                    "text": chunk,
                    "page": page["page"]
                })

            # Следующий фрагмент начинается с учётом перекрытия.
            start += chunk_size - overlap

    return chunks


chunks = chunk_text(
    pages=pages,
    chunk_size=CHUNK_SIZE,
    overlap=CHUNK_OVERLAP
)

print(f"Создано чанков: {len(chunks)}")
print("Пример чанка:")
print(chunks[0])

## 6. Подключение к OpenRouter

OpenRouter предоставляет OpenAI-совместимый API. Поэтому можно использовать клиент `OpenAI`, но указать другой `base_url`.

In [ ]:
# API-ключ вводится через getpass, чтобы не отображать его в ноутбуке.
OPENROUTER_API_KEY = getpass("Введите OPENROUTER_API_KEY: ")

# Создаём клиент для OpenAI-совместимого API OpenRouter.
openrouter_client = OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1"
)

## 7A. Эмбеддинги через OpenRouter

Эмбеддинг — это числовой вектор, который описывает смысл текста. Похожие по смыслу тексты должны иметь близкие векторы. Эта ячейка строит эмбеддинги для всех чанков через OpenRouter.

In [ ]:
def embed_texts_openrouter(
    texts: List[str],
    model: str = EMBEDDING_MODEL_OPENROUTER,
    batch_size: int = 64
) -> np.ndarray:
    """
    Создаёт эмбеддинги для списка текстов через OpenRouter.

    Батчинг нужен, чтобы отправлять тексты пачками, а не по одному.
    """
    all_embeddings = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]

        response = openrouter_client.embeddings.create(
            model=model,
            input=batch
        )

        batch_embeddings = [item.embedding for item in response.data]
        all_embeddings.extend(batch_embeddings)

        print(f"Обработано текстов: {min(i + batch_size, len(texts))}/{len(texts)}")

    embeddings = np.array(all_embeddings, dtype="float32")
    return embeddings


chunk_texts = [chunk["text"] for chunk in chunks]

# Если используется этот вариант, retrieval-запросы тоже должны кодироваться через OpenRouter.
embeddings = embed_texts_openrouter(chunk_texts)

print("Размерность матрицы эмбеддингов:", embeddings.shape)

## 7B. Fallback: эмбеддинги через Hugging Face

Эту ячейку можно запускать вместо 7A. Она строит эмбеддинги локально через модель `sentence-transformers/all-MiniLM-L6-v2`.

In [ ]:
from sentence_transformers import SentenceTransformer

# Компактная модель эмбеддингов, которая хорошо подходит для учебного демо.
hf_embedding_model = SentenceTransformer(EMBEDDING_MODEL_HF)

chunk_texts = [chunk["text"] for chunk in chunks]

# normalize_embeddings=True сразу нормализует векторы для cosine-подобного поиска.
embeddings = hf_embedding_model.encode(
    chunk_texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

print("Размерность матрицы эмбеддингов:", embeddings.shape)

## 8. Создание FAISS-индекса

FAISS хранит эмбеддинги чанков и быстро ищет ближайшие векторы. Это отвечает за retrieval-часть RAG.

In [ ]:
def build_faiss_index(embeddings: np.ndarray) -> faiss.Index:
    """
    Создаёт FAISS-индекс для поиска ближайших векторов.
    """
    dimension = embeddings.shape[1]

    # IndexFlatIP использует inner product.
    # После L2-нормализации inner product работает как cosine similarity.
    index = faiss.IndexFlatIP(dimension)

    # Нормализуем эмбеддинги перед добавлением в индекс.
    faiss.normalize_L2(embeddings)

    # Добавляем все векторы чанков в индекс.
    index.add(embeddings)

    return index


index = build_faiss_index(embeddings)

print("Количество векторов в индексе:", index.ntotal)

## 9. Retrieval: поиск релевантных чанков

Вопрос пользователя тоже превращается в эмбеддинг. Затем FAISS ищет чанки, чьи векторы ближе всего к вектору вопроса.

In [ ]:
def embed_query_openrouter(
    query: str,
    model: str = EMBEDDING_MODEL_OPENROUTER
) -> np.ndarray:
    """
    Создаёт эмбеддинг для вопроса через OpenRouter.
    """
    response = openrouter_client.embeddings.create(
        model=model,
        input=query
    )

    query_embedding = np.array(
        response.data[0].embedding,
        dtype="float32"
    ).reshape(1, -1)

    faiss.normalize_L2(query_embedding)

    return query_embedding


def embed_query_hf(query: str) -> np.ndarray:
    """
    Создаёт эмбеддинг для вопроса через локальную Hugging Face модель.
    """
    query_embedding = hf_embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    return query_embedding


def retrieve(
    query: str,
    index: faiss.Index,
    chunks: List[Dict[str, Any]],
    top_k: int = 5,
    embedding_backend: str = "openrouter"
) -> List[Dict[str, Any]]:
    """
    Находит top_k наиболее релевантных чанков для вопроса.

    embedding_backend:
    - "openrouter": использовать эмбеддинги OpenRouter;
    - "hf": использовать локальные эмбеддинги Hugging Face.
    """
    if embedding_backend == "openrouter":
        query_embedding = embed_query_openrouter(query)
    elif embedding_backend == "hf":
        query_embedding = embed_query_hf(query)
    else:
        raise ValueError("embedding_backend должен быть 'openrouter' или 'hf'.")

    # scores — оценки близости, indices — индексы найденных чанков.
    scores, indices = index.search(query_embedding, top_k)

    results = []

    for score, idx in zip(scores[0], indices[0]):
        chunk = chunks[idx].copy()
        chunk["score"] = float(score)
        chunk["chunk_id"] = int(idx)
        results.append(chunk)

    return results

In [ ]:
# Быстрый тест retrieval-части без генерации ответа.
test_query = "How does armor class work?"

retrieved_chunks = retrieve(
    query=test_query,
    index=index,
    chunks=chunks,
    top_k=TOP_K,
    embedding_backend="openrouter"  # замените на "hf", если использовали ячейку 7B
)

for item in retrieved_chunks:
    print("=" * 80)
    print(f"Chunk ID: {item['chunk_id']}")
    print(f"Page: {item['page']}")
    print(f"Score: {item['score']:.4f}")
    print(item["text"][:700])

## 10. Формирование контекста для LLM

Найденные чанки объединяются в один блок контекста. В каждый источник добавляется номер страницы и `chunk_id`, чтобы ответ можно было связать с исходными фрагментами.

In [ ]:
def build_context(retrieved_chunks: List[Dict[str, Any]]) -> str:
    """
    Собирает найденные чанки в единый контекст для языковой модели.
    """
    context_parts = []

    for i, chunk in enumerate(retrieved_chunks, start=1):
        context_parts.append(
            f"[Источник {i} | страница {chunk['page']} | chunk_id {chunk['chunk_id']}]\n"
            f"{chunk['text']}"
        )

    return "\n\n".join(context_parts)


context = build_context(retrieved_chunks)
print(context[:2000])

## 11A. Генерация ответа через OpenRouter

Модель получает вопрос и найденный контекст. Системный промпт ограничивает ответ только предоставленными источниками, чтобы снизить риск галлюцинаций.

In [ ]:
def answer_with_openrouter(
    question: str,
    retrieved_chunks: List[Dict[str, Any]],
    model: str = GENERATION_MODEL_OPENROUTER
) -> str:
    """
    Генерирует ответ через OpenRouter на основе найденных чанков.
    """
    context = build_context(retrieved_chunks)

    system_prompt = """
Вы — помощник преподавателя на занятии по RAG.
Отвечайте только на основе предоставленного контекста.
Если ответа нет в контексте, прямо скажите: "В найденном контексте нет достаточно информации для ответа".
Не выдумывайте правила.
В конце ответа укажите, на какие источники из контекста вы опирались.
""".strip()

    user_prompt = f"""
Вопрос:
{question}

Контекст:
{context}

Дайте краткий, точный и структурированный ответ.
""".strip()

    response = openrouter_client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.2
    )

    return response.choices[0].message.content


question = "How does armor class work?"

retrieved_chunks = retrieve(
    query=question,
    index=index,
    chunks=chunks,
    top_k=TOP_K,
    embedding_backend="openrouter"  # замените на "hf", если использовали HF embeddings
)

answer = answer_with_openrouter(question, retrieved_chunks)
print(answer)

## 11B. Fallback: генерация через Hugging Face

Эта ячейка показывает, что генератор можно заменить независимо от retrieval-части. Retrieval остаётся прежним, меняется только модель, которая формирует финальный ответ.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

# Компактная instruct-модель для учебного запуска.
# При наличии GPU и достаточной памяти можно заменить её на более сильную модель.
HF_GENERATION_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(HF_GENERATION_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    HF_GENERATION_MODEL,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

hf_generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=500,
    temperature=0.2,
    do_sample=True
)

In [ ]:
def answer_with_huggingface(
    question: str,
    retrieved_chunks: List[Dict[str, Any]]
) -> str:
    """
    Генерирует ответ локальной Hugging Face моделью.
    """
    context = build_context(retrieved_chunks)

    messages = [
        {
            "role": "system",
            "content": (
                "You are a teaching assistant for a RAG lesson. "
                "Answer only using the provided context. "
                "If the context does not contain enough information, say so. "
                "Do not invent rules."
            )
        },
        {
            "role": "user",
            "content": (
                f"Question:\n{question}\n\n"
                f"Context:\n{context}\n\n"
                "Give a concise answer and cite the source numbers from the context."
            )
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    output = hf_generator(prompt)[0]["generated_text"]

    # Возвращаем только новый текст, сгенерированный после prompt.
    return output[len(prompt):].strip()


question = "How does armor class work?"

retrieved_chunks = retrieve(
    query=question,
    index=index,
    chunks=chunks,
    top_k=TOP_K,
    embedding_backend="hf"  # используйте "openrouter", если эмбеддинги были через OpenRouter
)

answer = answer_with_huggingface(question, retrieved_chunks)
print(answer)

## 12. Полная RAG-функция

Эта функция объединяет все этапы: получает вопрос, ищет релевантные чанки, передаёт их генератору и возвращает ответ вместе со списком источников.

In [ ]:
def rag_answer(
    question: str,
    embedding_backend: str = "openrouter",
    generation_backend: str = "openrouter"
) -> Dict[str, Any]:
    """
    Полный RAG-пайплайн:
    1. Получить вопрос.
    2. Превратить вопрос в эмбеддинг.
    3. Найти похожие чанки в FAISS.
    4. Передать найденный контекст в LLM.
    5. Вернуть ответ и использованные источники.
    """
    retrieved_chunks = retrieve(
        query=question,
        index=index,
        chunks=chunks,
        top_k=TOP_K,
        embedding_backend=embedding_backend
    )

    if generation_backend == "openrouter":
        answer = answer_with_openrouter(question, retrieved_chunks)
    elif generation_backend == "hf":
        answer = answer_with_huggingface(question, retrieved_chunks)
    else:
        raise ValueError("generation_backend должен быть 'openrouter' или 'hf'.")

    return {
        "question": question,
        "answer": answer,
        "sources": [
            {
                "chunk_id": chunk["chunk_id"],
                "page": chunk["page"],
                "score": chunk["score"],
                "preview": chunk["text"][:300]
            }
            for chunk in retrieved_chunks
        ]
    }


result = rag_answer(
    question="What happens when a character gains a level?",
    embedding_backend="openrouter",
    generation_backend="openrouter"
)

print("ВОПРОС:")
print(result["question"])

print("\nОТВЕТ:")
print(result["answer"])

print("\nИСТОЧНИКИ:")
for source in result["sources"]:
    print(source)

## 13. Мини-интерфейс для демонстрации

Простой цикл позволяет задавать несколько вопросов подряд и смотреть, какие источники были найдены для каждого ответа.

In [ ]:
while True:
    question = input("\nВведите вопрос по книге или 'exit' для выхода: ")

    if question.lower().strip() in ["exit", "quit", "выход"]:
        break

    result = rag_answer(
        question=question,
        embedding_backend="openrouter",
        generation_backend="openrouter"
    )

    print("\nОтвет:")
    print(result["answer"])

    print("\nНайденные источники:")
    for source in result["sources"]:
        print(
            f"- page={source['page']}, "
            f"chunk_id={source['chunk_id']}, "
            f"score={source['score']:.4f}"
        )

## Что показывает это демо

RAG разделяет задачу на две части: поиск релевантного контекста и генерацию ответа. Retrieval извлекает подходящие фрагменты из корпуса, а генератор формирует ответ на их основе.

Эмбеддинги используются для поиска смысловой близости, а не для генерации текста. FAISS быстро ищет ближайшие векторы, но сам по себе не является языковой моделью.

Качество RAG зависит от качества чанкинга, embedding-модели, стратегии retrieval и промпта. Если нужный фрагмент не попал в top-k, генератор может не получить достаточно информации для корректного ответа.